# Mario Kart Video Preprocessor
This notebook renames raw videos, moves them to the processed folder, generates a label CSV template, and splits the video into frames for ML training.

In [3]:
import csv
import os
import shutil
from datetime import datetime

import cv2
from dotenv import find_dotenv, load_dotenv
from tqdm import tqdm

dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

# --- CONFIGURATION ---
# 1. Change these for each video you process
RAW_VIDEO_FILENAME = "2026-05-11 15-48-39.mov"  # Must exist in training_data/raw_videos/
TRACK_NAME = "rainbow_road"
PLAYERS = 1
RACE_TYPE = "standard_loop"  # choices: standard, knockout, time_trial
IS_ONLINE = False

# 2. Date/Time (Leave empty to use current UTC time)
MANUAL_DATE = "2026-05-11"  # Format: "MM_DD_YYYY" (e.g. "05_12_2026")
MANUAL_TIME = "15-48-39"  # Format: "HH_MM_SS"   (e.g. "22_15_00")

# 3. Path Settings
PROJECT_ROOT = os.path.dirname(dotenv_path) if dotenv_path else ".."

raw_dir_val = os.getenv("TRAINING_DATA_RAW_VIDEOS_DIRECTORY", "training_data/raw_videos")
RAW_DIR = (
    raw_dir_val
    if os.path.isabs(raw_dir_val)
    else os.path.abspath(os.path.join(PROJECT_ROOT, raw_dir_val))
)

video_dir_val = os.getenv("TRAINING_DATA_PROCESSED_VIDEOS_DIRECTORY", "training_data/videos")
VIDEO_DIR = (
    video_dir_val
    if os.path.isabs(video_dir_val)
    else os.path.abspath(os.path.join(PROJECT_ROOT, video_dir_val))
)

label_dir_val = os.getenv("TRAINING_DATA_LABELS_DIRECTORY", "training_data/labels")
LABEL_DIR = (
    label_dir_val
    if os.path.isabs(label_dir_val)
    else os.path.abspath(os.path.join(PROJECT_ROOT, label_dir_val))
)

frame_dir_val = os.getenv("TRAINING_DATA_FRAMES_DIRECTORY", "training_data/frames")
FRAME_DIR = (
    frame_dir_val
    if os.path.isabs(frame_dir_val)
    else os.path.abspath(os.path.join(PROJECT_ROOT, frame_dir_val))
)

# 4. Frame Settings
TARGET_WIDTH = 960
TARGET_HEIGHT = 540
FRAME_STRIDE = 1  # 1 = every frame, 2 = every other frame, etc.

In [ ]:
def generate_video_id():
    now = datetime.utcnow()
    date_str = MANUAL_DATE if MANUAL_DATE else now.strftime("%m_%d_%Y")
    time_str = MANUAL_TIME if MANUAL_TIME else now.strftime("%H_%M_%S")
    player_str = f"p{PLAYERS}"
    lobby_str = "online" if IS_ONLINE else "local"
    chat_str = "nochat"

    return f"mkw_{TRACK_NAME}_{player_str}_{RACE_TYPE}_{lobby_str}_{chat_str}_{date_str}_{time_str}"


def process_video():
    # 1. Validate Input
    raw_path = os.path.join(RAW_DIR, RAW_VIDEO_FILENAME)
    if not os.path.exists(raw_path):
        print(f"❌ Error: Raw video not found at {raw_path}")
        return

    video_id = generate_video_id()
    print(f"🚀 Starting processing for Video ID: {video_id}")

    # 2. Move and Rename Video
    processed_video_path = os.path.join(VIDEO_DIR, f"{video_id}.mp4")
    os.makedirs(VIDEO_DIR, exist_ok=True)
    shutil.move(raw_path, processed_video_path)
    print(f"✅ Video moved to: {processed_video_path}")

    # 3. Create Frame Directory
    run_frame_dir = os.path.join(FRAME_DIR, video_id)
    os.makedirs(run_frame_dir, exist_ok=True)

    # 4. Extract Frames and Prepare CSV rows
    cap = cv2.VideoCapture(processed_video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"📦 Extracting {total_frames} frames...")

    csv_headers = [
        "frame_id",
        "track",
        "placement",
        "lap_count",
        "coin_count",
        "primary_item",
        "secondary_item",
        "race_phase",
    ]
    csv_rows = []

    frame_idx = 0
    pbar = tqdm(total=total_frames, desc="processsing video")
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % FRAME_STRIDE == 0:
            frame_name = f"{frame_idx:06d}_{video_id}.png"
            frame_path = os.path.join(run_frame_dir, frame_name)

            # Resize for ML efficiency
            resized = cv2.resize(frame, (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_AREA)
            cv2.imwrite(frame_path, resized)

            csv_rows.append(
                {
                    "frame_id": frame_name,
                    "track": TRACK_NAME,
                    "placement": "",
                    "coin_count": "",
                    "primary_item": "",
                    "secondary_item": "",
                    "race_phase": "",
                }
            )

        frame_idx += 1
        # if frame_idx % 500 == 0:
        #     print(f"Processed {frame_idx}/{total_frames} frames...")
        pbar.update()
    pbar.close()

    cap.release()
    print(f"✅ {len(csv_rows)} frames saved to {run_frame_dir}")

    # 5. Save CSV Label Template
    os.makedirs(LABEL_DIR, exist_ok=True)
    csv_path = os.path.join(LABEL_DIR, f"UNFINISHED_{video_id}.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=csv_headers)
        writer.writeheader()
        writer.writerows(csv_rows)

    print(f"✅ Label template created at: {csv_path}")
    print("🎉 Done!")


process_video()

/var/folders/vf/22dqhxgd7v7bf9xyk1l00cs40000gn/T/ipykernel_24686/2475886156.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


🚀 Starting processing for Video ID: mkw_rainbow_road_p1_standard_loop_local_nochat_2026-05-11_15-48-39
✅ Video moved to: ../training_data/videos/mkw_rainbow_road_p1_standard_loop_local_nochat_2026-05-11_15-48-39.mp4
📦 Extracting 9339 frames...


processsing video: 100%|██████████| 9339/9339 [02:35<00:00, 59.90it/s]

✅ 9339 frames saved to ../training_data/frames/mkw_rainbow_road_p1_standard_loop_local_nochat_2026-05-11_15-48-39
✅ Label template created at: ../training_data/labels/mkw_rainbow_road_p1_standard_loop_local_nochat_2026-05-11_15-48-39.csv
🎉 Done!
